In [ ]:
pip install opencv-python scikit-learn scikit-image tqdm

In [ ]:
import os
import cv2
import numpy as np
from xml.etree import ElementTree as ET
from skimage.feature import hog
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import joblib
#Initianl model

# Parse the XML files
def parse_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text.lower()
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append((name, (xmin, ymin, xmax, ymax)))
    return objects


# Load the images and extract the HOG features
def load_data(image_folder, annotation_folder, resize_dim=(64, 128)):
    features = []
    labels = []

    for filename in tqdm(os.listdir(image_folder)):
        if not filename.endswith(".jpg"):
            continue

        image_path = os.path.join(image_folder, filename)
        annotation_path = os.path.join(annotation_folder, filename.replace('.jpg', '.xml'))

        if not os.path.exists(annotation_path):
            continue

        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        objects = parse_annotation(annotation_path)
        for name, (xmin, ymin, xmax, ymax) in objects:
            roi = img[ymin:ymax, xmin:xmax]
            roi = cv2.resize(roi, resize_dim)
            fd = hog(roi, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), visualize=False)

            # Label as weapon or not a weapon
            if name in ['knife', 'pistol']:
                features.append(fd)
                labels.append('weapon')
            else:
                features.append(fd)
                labels.append('no_weapon')

    return np.array(features), np.array(labels)


# Load the train/test images and annotations
print("Loading training data...")
X_train, y_train = load_data('/content/drive/MyDrive/SOHAS/train_dataset/images', '/content/drive/MyDrive/SOHAS/train_dataset/annotations')

print("Loading test data...")
X_test, y_test = load_data('/content/drive/MyDrive/SOHAS/test_dataset/images', '/content/drive/MyDrive/SOHAS/test_dataset/annotations')

# Filter only the weapon images for the classification
X_train_weapon = X_train[y_train == 'weapon']
y_train_weapon = y_train[y_train == 'weapon']
X_test_weapon = X_test[y_test == 'weapon']
y_test_weapon = y_test[y_test == 'weapon']


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the weapon detection model
print("Training KNN for weapon detection (binary)...")
knn_detector = KNeighborsClassifier(n_neighbors=3)
knn_detector.fit(X_train_scaled, y_train)

y_pred_binary = knn_detector.predict(X_test_scaled)

print("\n[Detection] Weapon vs Non-Weapon:")
print(classification_report(y_test, y_pred_binary, target_names=['No Weapon', 'Weapon']))

# Train the weapon classification model
print("Training KNN for knife vs pistol classification...")
knn_classifier = KNeighborsClassifier(n_neighbors=3)
knn_classifier.fit(X_train_weapon, y_train_weapon)

y_pred_weapon = knn_classifier.predict(X_test_weapon)

print("\n[Classification] Knife vs Pistol:")
print(classification_report(y_test_weapon, y_pred_weapon))

# Save the trained models
print("Saving the KNN weapon detection model...")
joblib.dump(knn_detector, '/content/drive/MyDrive/knn_weapon_detector_model.pkl')

print("Saving the KNN knife/pistol classifier model...")
joblib.dump(knn_classifier, '/content/drive/MyDrive/knn_weapon_classifier_model.pkl')

print("Models saved successfully.")


Loading training data...


100%|██████████| 5002/5002 [41:22<00:00,  2.02it/s]


Loading test data...


100%|██████████| 857/857 [06:01<00:00,  2.37it/s]


Training KNN for weapon detection (binary)...

[Detection] Weapon vs Non-Weapon:
              precision    recall  f1-score   support

   No Weapon       0.79      0.86      0.83       321
      Weapon       0.92      0.87      0.89       548

    accuracy                           0.87       869
   macro avg       0.85      0.87      0.86       869
weighted avg       0.87      0.87      0.87       869

Training KNN for knife vs pistol classification...

[Classification] Knife vs Pistol:
              precision    recall  f1-score   support

      weapon       1.00      1.00      1.00       548

    accuracy                           1.00       548
   macro avg       1.00      1.00      1.00       548
weighted avg       1.00      1.00      1.00       548

Saving the KNN weapon detection model...
Saving the KNN knife/pistol classifier model...
Models saved successfully.


In [1]:
import os
import cv2
import numpy as np
from xml.etree import ElementTree as ET
from skimage.feature import hog
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import joblib
#Augmentation train and test code

# The 2 classes that the model should treat a weapons based on the annotation classes
WEAPON_CLASSES = ['knife', 'pistol']

# Parse the xml files containing the annotations
def parse_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text.lower()
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        objects.append((name, (xmin, ymin, xmax, ymax)))
    return objects

# Apply augmentation to the training images
def augment_image(img):

    augmented = []

    # Horizontaly flip the training images
    flipped = cv2.flip(img, 1)
    augmented.append(flipped)

    # Rotate the images in positive and negative directions
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    for angle in [-15, 15]:
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        augmented.append(rotated)

    # Adjust the brightness of the images
    bright = cv2.convertScaleAbs(img, alpha=1.2, beta=30)
    augmented.append(bright)

    # Gaussian blur the train images
    blurred = cv2.GaussianBlur(img, (3, 3), 0)
    augmented.append(blurred)

    return augmented

# Load training images and use HOG to extract the features
def load_data(image_folder, annotation_folder, resize_dim=(64, 128), augment=False):
    features = []
    labels = []

    for filename in tqdm(os.listdir(image_folder)):
        if not filename.endswith(".jpg"):
            continue

        image_path = os.path.join(image_folder, filename)
        annotation_path = os.path.join(annotation_folder, filename.replace('.jpg', '.xml'))

        if not os.path.exists(annotation_path):
            continue

        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        objects = parse_annotation(annotation_path)
        for name, (xmin, ymin, xmax, ymax) in objects:
            roi = img[ymin:ymax, xmin:xmax]
            if roi.size == 0:
                continue
            roi = cv2.resize(roi, resize_dim)
            fd = hog(roi, orientations=9, pixels_per_cell=(8, 8),
                     cells_per_block=(2, 2), visualize=False)
            # Label name of either knife or pistol
            if name in WEAPON_CLASSES:
                label = name
                weapon_tag = 'weapon'
            else:
                label = 'no_weapon'
                weapon_tag = 'no_weapon'

            features.append(fd)
            labels.append(weapon_tag)


            if augment:
                augmented_rois = augment_image(roi)
                for aug_roi in augmented_rois:
                    fd_aug = hog(aug_roi, orientations=9, pixels_per_cell=(8, 8),
                                 cells_per_block=(2, 2), visualize=False)
                    features.append(fd_aug)
                    labels.append(weapon_tag)

    return np.array(features), np.array(labels)


# File paths for the train/test images and annotations
train_images = '/content/drive/MyDrive/SOHAS/train_dataset/images'
train_annots = '/content/drive/MyDrive/SOHAS/train_dataset/annotations'
test_images = '/content/drive/MyDrive/SOHAS/test_dataset/images'
test_annots = '/content/drive/MyDrive/SOHAS/test_dataset/annotations'

# Load the augmented images
print("Loading training data with augmentation...")
X_train, y_train = load_data(train_images, train_annots, augment=True)

print("Loading test data without augmentation...")
X_test, y_test = load_data(test_images, test_annots, augment=False)

# Extract the weapon only class data
def extract_weapon_class_data(image_folder, annotation_folder, resize_dim=(64, 128)):
    features = []
    labels = []

    for filename in tqdm(os.listdir(image_folder)):
        if not filename.endswith(".jpg"):
            continue

        image_path = os.path.join(image_folder, filename)
        annotation_path = os.path.join(annotation_folder, filename.replace('.jpg', '.xml'))

        if not os.path.exists(annotation_path):
            continue

        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        objects = parse_annotation(annotation_path)
        for name, (xmin, ymin, xmax, ymax) in objects:
            if name not in WEAPON_CLASSES:
                continue
            roi = img[ymin:ymax, xmin:xmax]
            if roi.size == 0:
                continue
            roi = cv2.resize(roi, resize_dim)
            fd = hog(roi, orientations=9, pixels_per_cell=(8, 8),
                     cells_per_block=(2, 2), visualize=False)
            features.append(fd)
            labels.append(name)

    return np.array(features), np.array(labels)

print("Extracting weapon-only data for classification...")
X_train_weapon, y_train_weapon = extract_weapon_class_data(train_images, train_annots)
X_test_weapon, y_test_weapon = extract_weapon_class_data(test_images, test_annots)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model for weapon detection
print("\nTraining KNN for weapon detection...")
knn_detector = KNeighborsClassifier(n_neighbors=3)
knn_detector.fit(X_train_scaled, y_train)

# Evalate the predictions
y_pred_binary = knn_detector.predict(X_test_scaled)

print("\n[Detection] Weapon vs Non-Weapon:")
print(classification_report(y_test, y_pred_binary, target_names=['No Weapon', 'Weapon']))

# Train for weapon classification
print("\nTraining KNN for knife vs pistol classification...")
knn_classifier = KNeighborsClassifier(n_neighbors=3)
knn_classifier.fit(X_train_weapon, y_train_weapon)

y_pred_weapon = knn_classifier.predict(X_test_weapon)

print("\n[Classification] Knife vs Pistol:")
print(classification_report(y_test_weapon, y_pred_weapon))

# Save the models
print("\nSaving models...")
joblib.dump(knn_detector, '/content/drive/MyDrive/knn_augmented_detector_model.pkl')
joblib.dump(knn_classifier, '/content/drive/MyDrive/knn_augmented_classifier_model.pkl')
print("Models saved successfully.")


Loading training data with augmentation...


100%|██████████| 5002/5002 [2:44:00<00:00,  1.97s/it]


Loading test data without augmentation...


100%|██████████| 857/857 [12:41<00:00,  1.13it/s]


Extracting weapon-only data for classification...


100%|██████████| 857/857 [00:19<00:00, 45.03it/s]



Training KNN for weapon detection...

[Detection] Weapon vs Non-Weapon:
              precision    recall  f1-score   support

   No Weapon       0.82      0.88      0.85       321
      Weapon       0.92      0.89      0.90       548

    accuracy                           0.88       869
   macro avg       0.87      0.88      0.87       869
weighted avg       0.88      0.88      0.88       869


Training KNN for knife vs pistol classification...

[Classification] Knife vs Pistol:
              precision    recall  f1-score   support

       knife       0.98      0.91      0.94       468
      pistol       0.61      0.88      0.72        80

    accuracy                           0.90       548
   macro avg       0.80      0.89      0.83       548
weighted avg       0.92      0.90      0.91       548


Saving models...
Models saved successfully.
